# `mudataoom` quickstart — out-of-memory multimodal `.h5mu`

`mudataoom.MuDataOOM` is a drop-in for `mudata.MuData` whose **per-modality `X` stays on disk**. Each modality is an `anndataoom.AnnDataOOM` (Rust-backed), so reading million-cell CITE-seq / 10x Multiome atlases doesn't blow up your RAM.

**Stack**

| Layer | Crate / pkg | Responsibility |
|---|---|---|
| Python wrapper | `mudataoom` | duck-types `mudata.MuData` |
| Rust binding (bundled `_backend.so`) | `mudata-rs/pymudata` | PyO3 wrappers, h5mu I/O |
| Rust core | `mudata-rs/mudata` | `MuData<B>` type, ExternalLink workspace |
| Per-modality lazy ops | `anndataoom` + `anndata-rs` | chunked normalize / log1p / PCA, backed `X` |

**What's covered**

| Section | Topic |
|---|---|
| 1 | Build a tiny CITE-seq `.h5mu` |
| 2 | Open it lazily with `mudataoom.read_h5mu` |
| 3 | Inspect modalities and joint shape |
| 4 | Run `anndataoom` chunked ops on one modality |
| 5 | Write the result back to a new `.h5mu` |
| 6 | Re-open with upstream `mudata` to prove round-trip |

## 1. Build a tiny CITE-seq `.h5mu`

We synthesise a 256-cell × (40 genes + 6 ADTs) CITE-seq toy with upstream `mudata` so we have something real to load.

In [1]:
import os, tempfile, numpy as np, pandas as pd
from anndata import AnnData
from mudata import MuData

rng = np.random.default_rng(0)
n_obs = 256
obs_names = [f'cell_{i:04d}' for i in range(n_obs)]

rna = AnnData(
    X=rng.poisson(2.0, size=(n_obs, 40)).astype(np.float32),
    obs=pd.DataFrame({'sample': ['A']*128 + ['B']*128}, index=obs_names),
    var=pd.DataFrame(index=[f'gene_{j:03d}' for j in range(40)]),
)
prot = AnnData(
    X=rng.normal(5, 1, size=(n_obs, 6)).astype(np.float32),
    obs=pd.DataFrame({'sample': ['A']*128 + ['B']*128}, index=obs_names),
    var=pd.DataFrame(index=[f'prot_{j:02d}' for j in range(6)]),
)
mdata = MuData({'rna': rna, 'prot': prot})
mdata.obs['cohort'] = ['young']*128 + ['old']*128
mdata.obsm['X_joint'] = rng.normal(size=(n_obs, 4)).astype(np.float32)

tmpdir = tempfile.mkdtemp(prefix='moom-tutorial-', dir=os.environ.get('TMPDIR', '/tmp'))
h5mu_path = f'{tmpdir}/cite_toy.h5mu'
mdata.write_h5mu(h5mu_path)
print(f'wrote {h5mu_path}')
print(f'   {os.stat(h5mu_path).st_size / 1024:.1f} KB on disk')

wrote /tmp/moom-tutorial-l725m5e_/cite_toy.h5mu
   144.9 KB on disk


/scratch/users/steorra/env/omicdev/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/scratch/users/steorra/env/omicdev/lib/python3.10/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
/scratch/users/steorra/env/omicdev/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: 

## 2. Open lazily with `mudataoom.read_h5mu`

No copy. The Rust core builds one HDF5 ExternalLink per modality in a temp directory; `anndataoom` then opens each of those as a backed `AnnDataOOM`.

Use it as a context manager so the temp files + file handles are released when you're done.

In [2]:
import mudataoom as moom

mdata_oom = moom.read_h5mu(h5mu_path)
print(mdata_oom)

MuDataOOM [out-of-memory · backed] n_obs × n_vars = 256 × 46, axis=0
  mod (2)
    rna: AnnDataOOM 256 × 40
    prot: AnnDataOOM 256 × 6


## 3. Inspect modalities and joint shape

`mdata['rna']` / `mdata['prot']` return real `anndataoom.AnnDataOOM` objects — every `.X` access streams from disk through the Rust I/O layer.

In [3]:
import anndataoom

print(f'shape  = {mdata_oom.shape}')
print(f'n_mod  = {mdata_oom.n_mod}')
print(f'axis   = {mdata_oom.axis}  (0 = cell-aligned, 1 = feature-aligned)')
print(f'source = {mdata_oom.source_h5mu}')
print()
for name, m in mdata_oom.mod.items():
    print(f'  {name}: {type(m).__name__}  shape={m.shape}')
    assert isinstance(m, anndataoom.AnnDataOOM)

# Joint obs / obsm — small, in memory:
print(f"\nfirst 5 joint obs_names: {mdata_oom.obs_names[:5]}")

shape  = (256, 46)
n_mod  = 2
axis   = 0  (0 = cell-aligned, 1 = feature-aligned)
source = /tmp/moom-tutorial-l725m5e_/cite_toy.h5mu

  rna: AnnDataOOM  shape=(256, 40)
  prot: AnnDataOOM  shape=(256, 6)

first 5 joint obs_names: ['cell_0000', 'cell_0001', 'cell_0002', 'cell_0003', 'cell_0004']


## 4. `anndataoom` chunked ops on a modality

Single-modality preprocessing runs through the same lazy/chunked operator chain as standalone `anndataoom`. Normalize + log1p compose as lazy descriptors — `X` isn't materialised until you slice it.

In [4]:
rna_oom = mdata_oom['rna']
anndataoom.chunked_normalize_total(rna_oom, target_sum=1e4)
anndataoom.chunked_log1p(rna_oom)

# Materialise a 4-cell sample to verify the chain ran end-to-end.
sample = np.asarray(rna_oom.X[:4])
print(f'sample shape: {sample.shape}')
print(f'log-norm row sums (should be ~1e4 pre-log): {sample.sum(1)}')

sample shape: (4, 40)
log-norm row sums (should be ~1e4 pre-log): [183.99396 206.14778 196.29337 195.77654]


## 5. Write the processed `MuDataOOM` to a new `.h5mu`

Each modality streams chunk-by-chunk through the Rust writer; joint metadata is carried verbatim from the source `.h5mu` via libhdf5-level group copies. Compression is forced to `gzip` so the output is openable by stock h5py (no need to install blosc filter plugins).

In [5]:
out_path = f'{tmpdir}/cite_processed.h5mu'
mdata_oom.write_h5mu(out_path)
print(f'wrote {out_path}  ({os.stat(out_path).st_size / 1024:.1f} KB)')
mdata_oom.close()

wrote /tmp/moom-tutorial-l725m5e_/cite_processed.h5mu  (114.6 KB)


## 6. Re-open with upstream `mudata` to confirm round-trip

Stock `mudata.read_h5mu` (no `mudataoom`, no Rust) can decode the output. The joint `obs` / `obsm` survive the round-trip.

In [6]:
import mudata as md

reloaded = md.read_h5mu(out_path)
print(reloaded)
print()
print(f"joint obs columns survived: {list(reloaded.obs.columns)}")
print(f"joint obsm survived:        {list(reloaded.obsm.keys())}")

MuData object with n_obs × n_vars = 256 × 46
  obs:	'cohort'
  obsm:	'X_joint'
  2 modalities
    rna:	256 × 40
      obs:	'sample'
    prot:	256 × 6
      obs:	'sample'

joint obs columns survived: ['rna:sample', 'prot:sample', 'cohort']
joint obsm survived:        ['X_joint', 'prot', 'rna']


/scratch/users/steorra/env/omicdev/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/scratch/users/steorra/env/omicdev/lib/python3.10/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## Cleanup

In [7]:
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print(f'cleaned up {tmpdir}')

cleaned up /tmp/moom-tutorial-l725m5e_
